# Bayesian Fitting with BUMPS — Display API

This notebook demonstrates the new display-oriented Bayesian workflow in
EasyReflectometry.  It covers the same ground as the original
``bayesian_bumps.ipynb`` tutorial, but uses the high-level ``project.display``
and ``fitter.display`` API that is now the recommended way to inspect fit
quality, parameter correlations, posterior pairs, marginal distributions, and
posterior-predictive checks.

What's new:

- ``fitter.display.fit.results()`` — one call for classical *and* Bayesian fit
  reports.
- ``fitter.display.fit.correlations()`` — parameter correlation table (and
  heatmap).
- ``fitter.display.posterior.pairs()`` — corner / pair plot with automatic
  parameter selection.
- ``fitter.display.posterior.distribution()`` — one-dimensional marginal
  posteriors.
- ``fitter.display.posterior.reflectivity(data)`` and
  ``fitter.display.posterior.sld_profile()`` — posterior-predictive plots with
  95 % credible bands.
- ``project.display.*`` — the same workflow through a ``Project``, using
  experiment names instead of raw data.

**Note**: Requires ``corner``, ``arviz``, ``matplotlib``, and ``scipy`` (install
via ``pip install easyreflectometry[bayesian]`` or
``pip install -e '.[dev]'``).


In [ ]:
import warnings

import matplotlib.pyplot as plt
from easyscience.fitting import AvailableMinimizers

from easyreflectometry.calculators import CalculatorFactory
from easyreflectometry.data.measurement import load
from easyreflectometry.fitting import MultiFitter
from easyreflectometry.model import Model
from easyreflectometry.model import PercentageFwhm
from easyreflectometry.sample import Layer
from easyreflectometry.sample import Material
from easyreflectometry.sample import Multilayer
from easyreflectometry.sample import Sample

warnings.filterwarnings('ignore')

print('All libraries imported successfully.')

## 1. Load and inspect experimental data

In [ ]:
data_path = 'example.ort'
data = load(data_path)
print('Data loaded with keys:', list(data.keys()))

qz = data['coords']['Qz_0'].values
r_data = data['data']['R_0'].values

plt.figure(figsize=(7, 4))
plt.semilogy(qz, r_data, 'o', label='Data')
plt.xlabel('Q / Å⁻¹')
plt.ylabel('Reflectivity')
plt.title('Experimental data (example.ort)')
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

## 2. Estimate starting parameters from the data

Before building the model, we estimate the film thickness from the Kiessig
fringe spacing and the background level from the high-Q noise floor.  This
gives the optimiser a much better starting point than arbitrary defaults.

In [ ]:
import numpy as np
from scipy.signal import find_peaks

# --- film thickness from Kiessig fringes (R·Q⁴ representation) ---
mask = (qz > 0.04) & (qz < 0.15)
q_fringe = qz[mask]
rq4 = r_data[mask] * q_fringe**4

peaks, _ = find_peaks(rq4, distance=6, prominence=np.std(rq4) * 0.3)
if len(peaks) >= 2:
    dq = np.diff(q_fringe[peaks])
    thickness_est = float(np.mean(2 * np.pi / dq))
else:
    thickness_est = 250.0  # fallback

# --- background from high-Q noise floor ---
hi_q = qz > 0.2
background_est = float(np.median(r_data[hi_q])) if np.any(hi_q) else 1e-6

print(f'Estimated film thickness : {thickness_est:.0f} Å')
print(f'Estimated background      : {background_est:.2e}')

## 3. Build the monolayer model (Si / Film / D₂O)

In [ ]:
si = Material(sld=2.07, isld=0.0, name='Si')
film = Material(sld=2.0, isld=0.0, name='Film')
d2o = Material(sld=6.36, isld=0.0, name='D2O')

si_layer = Layer(material=si, thickness=0.0, roughness=3.0, name='Si')
film_layer = Layer(
    material=film, thickness=thickness_est, roughness=3.0, name='Film'
)
d2o_layer = Layer(material=d2o, thickness=0.0, roughness=3.0, name='D2O')

sample = Sample(
    Multilayer(si_layer),
    Multilayer(film_layer),
    Multilayer(d2o_layer),
    name='Monolayer Sample',
)

resolution = PercentageFwhm(0.02)
model = Model(
    sample=sample,
    scale=1.0,
    background=background_est,
    resolution_function=resolution,
    name='Monolayer Model',
)

# Make key parameters free with realistic bounds
film_layer.thickness.fixed = False
film_layer.thickness.bounds = (100, 400)

film_layer.roughness.fixed = False
film_layer.roughness.bounds = (0.0, 10.0)

film.sld.fixed = False
film.sld.bounds = (0.5, 4.0)

model.scale.fixed = False
model.scale.bounds = (0.8, 1.2)

model.background.fixed = False
model.background.bounds = (1e-7, 1e-5)

print('Model created with free parameters:')
for p in model.get_parameters():
    if not p.fixed:
        print(f'  {p.name}: value={p.value}, bounds={p.bounds}')

## 4. Set up the calculator and fitter

In [ ]:
interface = CalculatorFactory()
interface.switch('refnx')
model.interface = interface

fitter = MultiFitter(model)
fitter.switch_minimizer(AvailableMinimizers.Bumps)

print('Fitter ready with minimizer:', fitter.easy_science_multi_fitter.minimizer.name)

## 5. Classical fit — with the display API

In [ ]:
analysed = fitter.fit(data)
print('Classical fit successful.')

# One call replaces manual chi² printing:
fitter.display.fit.results()

In [ ]:
# Visual check of the classical fit
r_model = analysed['R_0_model'].values

plt.figure(figsize=(8, 5))
plt.semilogy(qz, r_data, 'o', label='Data', alpha=0.7)
plt.semilogy(qz, r_model, '-', label='Classical BUMPS fit', linewidth=2)
plt.xlabel('Q / Å⁻¹')
plt.ylabel('Reflectivity')
plt.title('Classical fit before Bayesian sampling')
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

## 6. Bayesian MCMC sampling — high-level API

In [ ]:
# Run the DREAM sampler.  ``as_object=True`` returns a PosteriorResults
# instance instead of a raw dictionary.
posterior = fitter.sample(
    data,
    samples=2000,
    burn=500,
    thin=10,
    seed=42,
    as_object=True,
)

print(posterior)

## 7. Inspect results with the display API

The display API gives you four key views without manual plotting or dictionary unpacking.

### 7.1 Fit results — classical *and* Bayesian in one report

In [ ]:
fitter.display.fit.results()

### 7.2 Parameter correlations

In [ ]:
fitter.display.fit.correlations(precision=3)

### 7.3 Posterior pair plot (corner plot)

In [ ]:
fitter.display.posterior.pairs()

### 7.4 Marginal posterior distributions

In [ ]:
fitter.display.posterior.distribution()

## 8. Posterior-predictive checks

In [ ]:
fitter.display.posterior.reflectivity(data)

In [ ]:
fitter.display.posterior.sld_profile()

## 9. Project-level workflow

The same display API works at the project level.  The project owns fit and posterior state, so you don't need to hold on to the ``fitter`` variable.  ``Project.fit()`` and ``Project.sample()`` resolve ``data`` from the current experiment when omitted.

In [ ]:
from easyreflectometry import Project

proj = Project()
proj.default_model()
proj.models[0].interface = CalculatorFactory()

# Load a data file as an experiment
proj.load_new_experiment(data_path)

# Switch minimizer
proj.minimizer = AvailableMinimizers.Bumps

# Make the same parameters free
m = proj.models[0]
m.sample[1].layers[0].thickness.fixed = False
m.sample[1].layers[0].thickness.bounds = (100, 400)
m.sample[1].layers[0].material.sld.fixed = False
m.sample[1].layers[0].material.sld.bounds = (0.5, 4.0)
m.scale.fixed = False
m.scale.bounds = (0.8, 1.2)
m.background.fixed = False
m.background.bounds = (1e-7, 1e-5)

print('Project ready.')

In [ ]:
# Project.fit() and Project.sample() capture state for project.display
proj.fit()
proj.sample(samples=2000, burn=500, thin=10, seed=42)

In [ ]:
# Same display API, now at the project level:
proj.display.fit.results()
proj.display.fit.correlations(precision=3)

In [ ]:
proj.display.posterior.pairs()
proj.display.posterior.distribution()
proj.display.posterior.reflectivity()
proj.display.posterior.sld_profile()

## 10. Summary

The display API unifies the inspection workflow across fitter-only and
project-level usage.  Key points:

- ``fitter.display.*`` and ``project.display.*`` share the same method names.
- ``fit.results()`` shows classical *and* Bayesian information together.
- ``posterior.pairs()`` automatically selects the most informative parameters.
- ``posterior.reflectivity()`` at the project level uses experiment names
  (or defaults to the current experiment).
- The underlying ``PosteriorResults`` object is still available for
  lower-level access when needed.

The API surface demonstrated:

| Method | What it shows |
|---|---|
| ``display.fit.results()`` | Classical + Bayesian fit report |
| ``display.fit.correlations()`` | Parameter correlation table |
| ``display.posterior.pairs()`` | Corner / pair plot |
| ``display.posterior.distribution()`` | 1D marginal posteriors |
| ``display.posterior.reflectivity()`` | Predictive reflectivity + 95% band |
| ``display.posterior.sld_profile()`` | Predictive SLD + 95% band |
